# Phase 1: GEE Multi-Modal Planetary Harvester
### SPARRSO Fellowship: Single-Date SAR-Optical Cloud Removal Dataset Preparation
**Key Improvements Applied:**
1. **AOI-Adaptive CRS:** Automatically selects `EPSG:32645` (UTM 45N) for Lon < 90°E (Khulna, Rangpur) and `EPSG:32646` (UTM 46N) for Lon >= 90°E (Sylhet, Chittagong).
2. **Strict Channels:** 4 Optical (B2, B3, B4, B8) + 2 SAR (VV, VH in GEE native dB).
3. **Quality Mask:** S2 Cloud Probability kept separate as a quality/loss mask, NOT model input.
4. **Proxy Target vs. Synthetic Ground Truth Strategy.
5. **Explicit ee.Image Casting:** Prevents FeatureCollection 'addBands is not a function' runtime error.

In [ ]:
import ee
import geemap
import time

# Initialize Earth Engine
try:
    ee.Initialize()
    print("[✓] Earth Engine Initialized Successfully!")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("[✓] Earth Engine Authenticated and Initialized!")

In [ ]:
# Define Bangladesh Study Region (Choose Khulna or Sylhet)
REGION_NAME = "Khulna_Delta"
ROI_COORDS = [89.35, 22.30, 89.65, 22.60]  # [min_lon, min_lat, max_lon, max_lat]
roi = ee.Geometry.Rectangle(ROI_COORDS)

# Determine AOI-specific CRS (90°E divider)
TARGET_CRS = 'EPSG:32646' if ROI_COORDS[0] >= 90.0 else 'EPSG:32645'
print(f"[*] Region: {REGION_NAME} | Longitude: {ROI_COORDS[0]}°E")
print(f"[*] Assigned Projected CRS: {TARGET_CRS}")

# Monsoon Date Window
START_DATE = '2024-06-01'
END_DATE = '2024-09-30'

In [ ]:
# 1. Query Cloudy Sentinel-2 L2A and join with Cloud Probability
OPTICAL_BANDS = ['B2', 'B3', 'B4', 'B8']  # 10m bands

s2_sr = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.gte('CLOUDY_PIXEL_PERCENTAGE', 20))
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 85))
)

s2_cloud_prob = (
    ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
)

join_filter = ee.Filter.equals(leftField='system:index', rightField='system:index')
s2_joined = ee.Join.inner('primary', 'prob').apply(s2_sr, s2_cloud_prob, join_filter)

def process_optical(feature):
    prim = ee.Image(feature.get('primary'))
    prob = ee.Image(feature.get('prob')).select('probability').rename('cloud_prob')
    mask = prob.gte(40).rename('cloud_mask')
    
    # Scale DN by 1/10000 to get physical surface reflectance units
    scaled_opt = prim.select(OPTICAL_BANDS).multiply(0.0001)
    return scaled_opt.addBands([prob, mask]).copyProperties(prim, prim.propertyNames())

cloudy_s2_col = ee.ImageCollection(s2_joined.map(process_optical))
print(f"[✓] Found {cloudy_s2_col.size().getInfo()} cloudy Sentinel-2 scenes.")

In [ ]:
# 2. Query Sentinel-1 GRD (IW Mode, VV + VH, natively in calibrated dB)
SAR_BANDS = ['VV', 'VH']

s1_col = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(SAR_BANDS)
)
print(f"[✓] Found {s1_col.size().getInfo()} Sentinel-1 SAR scenes.")

In [ ]:
# 3. Spatio-Temporal Join: Pair each cloudy S2 image with nearest S1 SAR (Δt <= 3 days)
MAX_DIFF_MS = 3 * 24 * 3600 * 1000  # 3 days in milliseconds

t_filter = ee.Filter.maxDifference(MAX_DIFF_MS, 'system:time_start', 'system:time_start')
paired_collection = ee.Join.saveBest('sar', 'delta_ms').apply(cloudy_s2_col, s1_col, t_filter)

def combine_pair(feature):
    # 1. Explicitly cast incoming feature to ee.Image to unlock .addBands and .clip
    s2_img = ee.Image(feature)
    # 2. Extract and cast the paired SAR image
    s1_img = ee.Image(feature.get('sar'))
    # 3. Compute absolute time gap in hours
    d_hrs = ee.Number(feature.get('delta_ms')).divide(3600000).abs()
    # 4. Fuse optical and SAR bands
    return s2_img.addBands(s1_img).clip(roi).set({
        's2_time': s2_img.get('system:time_start'),
        's1_time': s1_img.get('system:time_start'),
        'delta_hours': d_hrs,
        'cloudy_pct': s2_img.get('CLOUDY_PIXEL_PERCENTAGE')
    })

fused_pairs = ee.ImageCollection(paired_collection.map(combine_pair))
num_pairs = fused_pairs.size().getInfo()
print(f"[✓] Found {num_pairs} synchronized S1-S2 pairs (Δt <= 3 days)!")

In [ ]:
# 4. Interactive Visualization with geemap
if num_pairs > 0:
    sample_pair = ee.Image(fused_pairs.first())
    
    Map = geemap.Map()
    Map.centerObject(roi, zoom=12)
    
    # Optical True Color (B4-Red, B3-Green, B2-Blue), reflectance scaled [0, 0.3]
    opt_vis = {'bands': ['B4', 'B3', 'B2'], 'min': 0.0, 'max': 0.3, 'gamma': 1.2}
    # SAR Backscatter VV (dB range [-25, 0])
    sar_vis = {'bands': ['VV'], 'min': -25.0, 'max': 0.0, 'palette': ['black', 'gray', 'white']}
    # Cloud probability mask
    mask_vis = {'bands': ['cloud_mask'], 'min': 0, 'max': 1, 'palette': ['transparent', 'red']}
    
    Map.addLayer(sample_pair, opt_vis, 'Cloudy Sentinel-2 True Color')
    Map.addLayer(sample_pair, sar_vis, 'Sentinel-1 SAR VV (dB)')
    Map.addLayer(sample_pair, mask_vis, 'Detected Cloud Mask (40% Cutoff)', opacity=0.5)
    Map.addLayer(roi, {'color': 'blue'}, 'Study AOI Boundary')
    display(Map)
else:
    print("[!] No pairs found within the selected date window. Try expanding the date range.")

In [ ]:
# 5. Export Multi-Band GeoTIFF to Google Drive (Clean 10m grid in TARGET_CRS)
# Channels exported: B2, B3, B4, B8 (Optical), cloud_prob, cloud_mask, VV, VH (SAR)
if num_pairs > 0:
    first_pair = ee.Image(fused_pairs.first())
    meta = first_pair.getInfo()['properties']
    date_str = time.strftime('%Y%m%d', time.gmtime(meta['s2_time'] / 1000.0))
    
    export_task = ee.batch.Export.image.toDrive(
        image=first_pair,
        description=f"{REGION_NAME}_S1_S2_{date_str}",
        folder="SPARRSO_Dataset_Pairs",
        fileNamePrefix=f"{REGION_NAME}_S1_S2_{date_str}",
        scale=10,
        crs=TARGET_CRS,
        region=roi,
        maxPixels=1e9
    )
    export_task.start()
    print(f"[+] Export Task Dispatched: {REGION_NAME}_S1_S2_{date_str} to Drive (10m grid, {TARGET_CRS})")
    print("    Check task status in https://code.earthengine.google.com/tasks")